## Instalação de Bibliotecas

In [1]:
!pip install -q "transformers>=5.0.0" accelerate bitsandbytes peft trl datasets "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.2 MB/s eta 0:00:00


## Importações

In [2]:
from datasets import load_dataset
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, TaskType
from trl import SFTTrainer, SFTConfig
import torch, gc
import time

## Carregando o Modelo

In [3]:
REPO = "nvidia/Nemotron-Labs-Diffusion-3B"
OUTPUT_DIR = "./nemotron_diffusion_lora"

In [4]:
print("Carregando tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(REPO, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Carregando modelo para treino...")
model = AutoModel.from_pretrained(
    REPO,
    trust_remote_code=True,
    device_map="auto",
    cache_dir="./cache",
)
model.config.use_cache = True

Carregando tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.32k [00:00<?, ?B/s]

configuration_nemotron_labs_diffusion.py:   0%|          | 0.00/8.27k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/nvidia/Nemotron-Labs-Diffusion-3B:
- configuration_nemotron_labs_diffusion.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
/usr/local/lib/python3.12/dist-packages/transformers/modeling_rope_utils.py:1036: FutureWarning: `rope_config_validation` is deprecated and has been removed. Its functionality has been moved to RotaryEmbeddingConfigMixin.validate_rope method. PreTrainedConfig inherits this class, so please call self.validate_rope() instead. Also, make sure to use the new rope_parameters syntax. You can call self.standardize_rope_params() in the meantime.
  warnings.warn(
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'llama_4_scaling_beta'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'llama_4_scaling_beta'}
[transformer

...

[transformers] A new version of the following files was downloaded from https://huggingface.co/nvidia/Nemotron-Labs-Diffusion-3B:
- modeling_ministral.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] A new version of the following files was downloaded from https://huggingface.co/nvidia/Nemotron-Labs-Diffusion-3B:
- modeling_nemotron_labs_diffusion.py
- modeling_ministral.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] The `check_model_inputs` decorator is deprecated in favor of `merge_with_config_defaults`.


[ERROR] `cache_position` is part of Ministral3Model.forward's signature, but not documented. Make sure to add it to the docstring of the function in /root/.cache/huggingface/modules/transformers_modules/nvidia/Nemotron_hyphen_Labs_hyphen_Diffusion_hyphen_3B/0d51902da1f8869f83413ce642fab402fa5641e0/modeling_ministral.py.


model.safetensors:   0%|          | 0.00/7.66G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/237 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/133 [00:00<?, ?B/s]

In [5]:
original_forward = model.forward

def patched_forward(*args, **kwargs):
    outputs = original_forward(*args, **kwargs)
    if isinstance(outputs.loss, (tuple, list)):
        outputs = type(outputs)(loss=outputs.loss[0], **{k: v for k, v in outputs.items() if k != "loss"})
    return outputs

model.forward = patched_forward

## Carregando dados do Dataset

In [6]:
print("Carregando dataset...")
dataset = load_dataset("dominguesm/alpaca-data-pt-br", split="train[:200]")

def formatar_prompt(ex):
    user_msg = f"{ex['instruction']}\n{ex['input']}".strip() if ex.get('input') else ex['instruction']
    messages = [
        {"role": "system",    "content": "Você é um assistente prestativo que responde em português."},
        {"role": "user",      "content": user_msg},
        {"role": "assistant", "content": ex['output']},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

dataset = dataset.map(formatar_prompt)

Carregando dataset...


README.md:   0%|          | 0.00/12.0k [00:00<?, ?B/s]

data/train-00000-of-00001-63604ec441480f(…):   0%|          | 0.00/11.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51759 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

## Fine-Tuning do Modelo

In [7]:
lora_config = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05, bias="none",
    task_type=TaskType.CAUSAL_LM,
)

In [8]:
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    warmup_steps=10,
    lr_scheduler_type="cosine",
    report_to="none",
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
    args=sft_config,
)

print("Iniciando fine-tuning...")
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\n✅ Fine-tuning concluído! Adaptador salvo em '{OUTPUT_DIR}'")

Adding EOS to train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 11}.


Iniciando fine-tuning...


Step,Training Loss
10,1903.826172
20,1518.604102
30,1548.092383
40,1063.257617
50,1485.380176


/usr/local/lib/python3.12/dist-packages/transformers/modeling_rope_utils.py:1036: FutureWarning: `rope_config_validation` is deprecated and has been removed. Its functionality has been moved to RotaryEmbeddingConfigMixin.validate_rope method. PreTrainedConfig inherits this class, so please call self.validate_rope() instead. Also, make sure to use the new rope_parameters syntax. You can call self.standardize_rope_params() in the meantime.
  warnings.warn(
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'llama_4_scaling_beta'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'llama_4_scaling_beta'}
/usr/local/lib/python3.12/dist-packages/transformers/modeling_rope_utils.py:1036: FutureWarning: `rope_config_validation` is deprecated and has been removed. Its functionality has been moved to RotaryEmbeddingConfigMixin.validate_rope method. PreTrainedConfig inherits this class, so please call self.validate_rope() instead. Also, mak


✅ Fine-tuning concluído! Adaptador salvo em './nemotron_diffusion_lora'


## Avaliação do modelo

In [9]:
FRASES = [
    "O que é inteligência artificial?",
    "Como funciona a fotossíntese?",
    "Explique o conceito de inflação de forma simples.",
    "Quais são as principais linguagens de programação?",
    "O que é um buraco negro?",
]

SYSTEM_PROMPT = "Você é um assistente prestativo que responde em português."
MAX_NEW_TOKENS = 256
BLOCK_LENGTH   = 32

if MAX_NEW_TOKENS % BLOCK_LENGTH != 0:
    MAX_NEW_TOKENS = (MAX_NEW_TOKENS // BLOCK_LENGTH) * BLOCK_LENGTH

In [10]:
def gerar(pergunta, mode):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": pergunta},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    n_prompt_tokens = prompt_ids.shape[1]

    inicio = time.time()
    with torch.no_grad():
        if mode == "ar":
            out_ids, nfe = model.ar_generate(
                prompt_ids,
                max_new_tokens=MAX_NEW_TOKENS,
            )
        elif mode == "spec":
            out_ids, nfe = model.linear_spec_generate(
                prompt_ids,
                max_new_tokens=MAX_NEW_TOKENS,
                block_length=BLOCK_LENGTH,
                eos_token_id=tokenizer.eos_token_id,
            )
        else:  # dlm
            out_ids, nfe = model.generate(
                prompt_ids,
                max_new_tokens=MAX_NEW_TOKENS,
                block_length=BLOCK_LENGTH,
                threshold=0.9,
                eos_token_id=tokenizer.eos_token_id,
            )
    tempo = time.time() - inicio

    n_tokens_gerados = out_ids.shape[1] - n_prompt_tokens
    resposta = tokenizer.batch_decode(
        out_ids[:, n_prompt_tokens:], skip_special_tokens=True
    )[0].strip()

    return {
        "resposta":  resposta,
        "nfe":       nfe,
        "tempo":     round(tempo, 2),
        "tokens":    n_tokens_gerados,
        "tok_seg":   round(n_tokens_gerados / tempo, 1),
    }

In [11]:
model.eval()
model.config.use_cache = True

MODOS = ["ar", "dlm", "spec"]
resultados = []

for frase in FRASES:
    print(f"\n{'='*70}")
    print(f"📝 Pergunta: {frase}")
    print(f"{'='*70}")
    linha = {"pergunta": frase}

    for modo in MODOS:
        print(f"\n  ⏳ Gerando [{modo.upper()}]...")
        r = gerar(frase, modo)
        linha[modo] = r
        print(f"  🤖 [{modo.upper()}] {r['resposta'][:120]}{'...' if len(r['resposta']) > 120 else ''}")
        print(f"       NFE={r['nfe']} | Tempo={r['tempo']}s | Tokens={r['tokens']} | {r['tok_seg']} tok/s")

    resultados.append(linha)


📝 Pergunta: O que é inteligência artificial?

  ⏳ Gerando [AR]...
  🤖 [AR] A inteligência artificial (IA) é um campo da ciência da computação que se concentra em criar sistemas capazes de realiza...
       NFE=199 | Tempo=24.42s | Tokens=199 | 8.1 tok/s

  ⏳ Gerando [DLM]...
  🤖 [DLM] A inteligência artificial (IA) é uma área da ciência da computação que busca sim criar sistemas capazes de realizar tare...
       NFE=79 | Tempo=20.48s | Tokens=88 | 4.3 tok/s

  ⏳ Gerando [SPEC]...
  🤖 [SPEC] A **inteligência artificial (IA)** é um campo da ciência da computação que se concentra em criar sistemas capazes de rea...
       NFE=179 | Tempo=41.27s | Tokens=257 | 6.2 tok/s

📝 Pergunta: Como funciona a fotossíntese?

  ⏳ Gerando [AR]...
  🤖 [AR] A **fotossíntese** é o processo pelo qual as plantas, algas e algumas bactérias (como cianobactérias) convertem a energi...
       NFE=256 | Tempo=12.87s | Tokens=256 | 19.9 tok/s

  ⏳ Gerando [DLM]...
  🤖 [DLM] A fototossíntese** é um processo funda

In [12]:
print(f"\n\n{'='*70}")
print("📊 RESUMO COMPARATIVO")
print(f"{'='*70}")
print(f"{'Pergunta':<35} {'Modo':<6} {'NFE':>6} {'Tempo':>7} {'Tokens':>7} {'tok/s':>7}")
print("-"*70)

for linha in resultados:
    for i, modo in enumerate(MODOS):
        r = linha[modo]
        pergunta_str = linha['pergunta'][:33] + ".." if i == 0 else " " * 35
        print(f"{pergunta_str:<35} {modo.upper():<6} {r['nfe']:>6} {r['tempo']:>6}s {r['tokens']:>7} {r['tok_seg']:>7}")
    print("-"*70)

# Médias por modo
print(f"\n{'Médias por modo':^70}")
print(f"{'Modo':<6} {'NFE médio':>10} {'Tempo médio':>12} {'Tokens médio':>13} {'tok/s médio':>12}")
print("-"*55)
for modo in MODOS:
    nfes    = [r[modo]['nfe']     for r in resultados]
    tempos  = [r[modo]['tempo']   for r in resultados]
    tokens  = [r[modo]['tokens']  for r in resultados]
    toks    = [r[modo]['tok_seg'] for r in resultados]
    print(f"{modo.upper():<6} {sum(nfes)/len(nfes):>10.1f} {sum(tempos)/len(tempos):>11.2f}s {sum(tokens)/len(tokens):>13.1f} {sum(toks)/len(toks):>12.1f}")



📊 RESUMO COMPARATIVO
Pergunta                            Modo      NFE   Tempo  Tokens   tok/s
----------------------------------------------------------------------
O que é inteligência artificial?..  AR        199  24.42s     199     8.1
                                    DLM        79  20.48s      88     4.3
                                    SPEC      179  41.27s     257     6.2
----------------------------------------------------------------------
Como funciona a fotossíntese?..     AR        256  12.87s     256    19.9
                                    DLM       219  50.07s     256     5.1
                                    SPEC      169  37.96s     256     6.7
----------------------------------------------------------------------
Explique o conceito de inflação d.. AR        256  12.78s     256    20.0
                                    DLM       234  52.53s     256     4.9
                                    SPEC      173  38.86s     258     6.6
------------------------